# 08. Query Translation: Asking Better Questions for Better Retrieval

> **Promise:** By the end of this notebook, you will be able to implement Multi-Query expansion, RAG-Fusion with Reciprocal Rank Fusion (RRF), query decomposition, step-back prompting, and Hypothetical Document Embeddings (HyDE) to overcome ambiguous user queries.

### Learning Objectives
- Understand why raw user queries fail and expand them using Multi-Query perspective generation.
- Master the mathematics of Reciprocal Rank Fusion (RRF) with a hand-worked numerical example and code implementation.
- Implement advanced query translation patterns: Decomposition, Step-Back prompting, and HyDE, evaluating their latency vs recall trade-offs.

### Prerequisites
- Notebook 05 (Retrieval)
- Notebook 06 (Generation & Chains)


## 1. Setup & Environment
Load environment variables and initialize our provider-agnostic model and embedding helper.

In [1]:
import os, sys
from pathlib import Path

# Add project root to path for shared utilities
sys.path.insert(0, str(Path.cwd().parent.parent if Path.cwd().name == "notebooks" else Path.cwd()))
from course.utils.models import get_llm, get_embeddings, detect_provider
from course.utils.helpers import get_data_dir

llm = get_llm(temperature=0.0)
embeddings = get_embeddings()
data_dir = get_data_dir()
print(f"Active Provider: {detect_provider()}")
print(f"Data Directory: {data_dir}")

Active Provider: local
Data Directory: C:\project 03\llm\course\data


## 2. Concept & Intuition

### Why Do Raw User Queries Fail?

In real-world applications, user queries rarely match the exact terminology stored in your vector database:
1. **Vocabulary Mismatch**: A user asks *"How do I fix network drops?"*, but the engineering manual documents *"ERR_QUANTUM_TIMEOUT UDP socket buffer overflow"*.
2. **Under-specified Queries**: A user asks *"What is the policy?"* without specifying which office, department, or topic.
3. **Multi-Part Questions**: A user asks *"How much is the Enterprise plan and what ROS2 versions work with it?"*, which requires finding two unrelated chunks in different files.

**Query Translation** solves this by using an LLM to rewrite, expand, or decompose the raw question before querying the vector store.

```
                              ┌──> Query 1: Technical Perspective ──> Retriever ┐
                              │                                                  │
[Raw User Query] ──> [LLM] ──┼──> Query 2: Operational Perspective ──> Retriever ┼──> [RRF Ranker] ──> [LLM Answer]
                              │                                                  │
                              └──> Query 3: Architectural Perspective ──> Retriever ┘
```

## 3. Step-by-Step Hands-On Build

### Step 1: Ingesting Documents into an In-Memory Vector Store
Let's load our company documents from `course/data/` and index them into an `InMemoryVectorStore` to serve as our retrieval testbed.

In [2]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.vectorstores import InMemoryVectorStore

docs = []
for p in [data_dir / "company_handbook.txt", data_dir / "product_faq.md", data_dir / "recent_changelog.md"]:
    docs.extend(TextLoader(str(p), encoding="utf-8").load())

splitter = RecursiveCharacterTextSplitter(chunk_size=380, chunk_overlap=40)
chunks = splitter.split_documents(docs)
vector_store = InMemoryVectorStore.from_documents(chunks, embeddings)
print(f"Indexed {len(chunks)} chunks into InMemoryVectorStore.")

C:\Users\Rahat\AppData\Local\Temp\ipykernel_25032\129095541.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader
C:\project 03\llm\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Indexed 35 chunks into InMemoryVectorStore.


> **What just happened?**
> We have an indexed vector store containing our fictional company knowledge base.

### Step 2: Pattern 1: Multi-Query Expansion
Multi-Query generates 3 distinct perspectives on the user's question, retrieves top chunks for each, and unions the unique results.

In [3]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

multi_query_prompt = ChatPromptTemplate.from_template(
    "You are an AI assistant. Generate 3 distinct search queries related to this question:\n"
    "Question: {question}\n"
    "Output 3 numbered queries, one per line."
)

multi_query_chain = multi_query_prompt | llm | StrOutputParser()
raw_variations = multi_query_chain.invoke({"question": "How do I resolve connection drops?"})
queries = [q.split(". ", 1)[-1].strip() for q in raw_variations.strip().split("\n") if q.strip()]
print("Generated Query Perspectives:")
for i, q in enumerate(queries, 1):
    print(f" {i}. {q}")

Generated Query Perspectives:
 1. What are the system specifications and features?
 2. What are the operational parameters and pricing?
 3. What are the technical troubleshooting steps?


> **What just happened?**
> The model produced 3 alternative search queries from different angles.

### Step 3: Retrieving and Unioning Chunks across Multi-Queries
Now we retrieve the top-2 chunks for each generated query and union them into a deduplicated set of candidate documents.

In [4]:
all_retrieved = []
for q in queries:
    results = vector_store.similarity_search(q, k=2)
    all_retrieved.extend(results)

# Deduplicate by document content
unique_docs = {doc.page_content: doc for doc in all_retrieved}.values()
print(f"Total retrieved: {len(all_retrieved)} chunks -> Unique deduplicated: {len(unique_docs)} chunks.")
for i, doc in enumerate(list(unique_docs)[:2], 1):
    print(f"\n--- Unique Chunk #{i} ---")
    print(doc.page_content[:150].strip() + "...")

Total retrieved: 6 chunks -> Unique deduplicated: 6 chunks.

--- Unique Chunk #1 ---
3. EXPENSES, TRAVEL & LEARNING STIPEND
- Meal Expense Limit: Employees traveling on company business are reimbursed up to $150 per day for meals witho...

--- Unique Chunk #2 ---
### What is QuantumCache v4?
QuantumCache v4 is a distributed, vector-indexed in-memory cache used by Novasphere orchestrators to cache navigational p...


> **What just happened?**
> Multi-query union broadened our recall without returning duplicate text.

### Step 4: Pattern 2: The Mathematics of Reciprocal Rank Fusion (RRF)
RRF combines ranked lists from multiple queries by computing a reciprocal rank score: Score(d) = sum(1 / (k + rank_i)). Let's study a hand-worked example before coding.

In [5]:
rrf_explanation = """
[RRF HAND-WORKED NUMERICAL EXAMPLE]
Suppose 2 queries retrieve 3 documents (Doc A, Doc B, Doc C) with constant k = 60:
- Query 1 ranks: Doc A (#1), Doc B (#2), Doc C (#3)
- Query 2 ranks: Doc B (#1), Doc C (#2), Doc A (#3)

RRF Calculation:
- Doc A Score: 1/(60 + 1) + 1/(60 + 3) = 1/61 + 1/63 = 0.01639 + 0.01587 = 0.03226
- Doc B Score: 1/(60 + 2) + 1/(60 + 1) = 1/62 + 1/61 = 0.01613 + 0.01639 = 0.03252
- Doc C Score: 1/(60 + 3) + 1/(60 + 2) = 1/63 + 1/62 = 0.01587 + 0.01613 = 0.03200

Result: Doc B wins! Even though Doc A was #1 in Query 1, Doc B had consensus near the top in both!
"""
print(rrf_explanation)


[RRF HAND-WORKED NUMERICAL EXAMPLE]
Suppose 2 queries retrieve 3 documents (Doc A, Doc B, Doc C) with constant k = 60:
- Query 1 ranks: Doc A (#1), Doc B (#2), Doc C (#3)
- Query 2 ranks: Doc B (#1), Doc C (#2), Doc A (#3)

RRF Calculation:
- Doc A Score: 1/(60 + 1) + 1/(60 + 3) = 1/61 + 1/63 = 0.01639 + 0.01587 = 0.03226
- Doc B Score: 1/(60 + 2) + 1/(60 + 1) = 1/62 + 1/61 = 0.01613 + 0.01639 = 0.03252
- Doc C Score: 1/(60 + 3) + 1/(60 + 2) = 1/63 + 1/62 = 0.01587 + 0.01613 = 0.03200

Result: Doc B wins! Even though Doc A was #1 in Query 1, Doc B had consensus near the top in both!



> **What just happened?**
> RRF rewards consensus across multiple search perspectives rather than relying on raw similarity numbers.

### Step 5: Implementing Reciprocal Rank Fusion in Python
Let's implement the `reciprocal_rank_fusion` algorithm and rank the chunks retrieved across our multi-queries.

In [6]:
def reciprocal_rank_fusion(results_by_query: list[list], k: int = 60) -> list[tuple]:
    scores = {}
    doc_map = {}
    for query_results in results_by_query:
        for rank, doc in enumerate(query_results, start=1):
            content = doc.page_content
            doc_map[content] = doc
            scores[content] = scores.get(content, 0.0) + (1.0 / (k + rank))
    ranked = sorted(scores.items(), key=lambda item: item[1], reverse=True)
    return [(doc_map[content], score) for content, score in ranked]

# Gather search results per query
results_by_query = [vector_store.similarity_search(q, k=3) for q in queries]
fused_results = reciprocal_rank_fusion(results_by_query, k=60)
print(f"RRF scored {len(fused_results)} unique documents across {len(queries)} queries:")
for doc, score in fused_results[:3]:
    print(f" RRF Score: {score:.5f} -> {doc.page_content[:90].strip()}...")

RRF scored 9 unique documents across 3 queries:
 RRF Score: 0.01639 -> 3. EXPENSES, TRAVEL & LEARNING STIPEND
- Meal Expense Limit: Employees traveling on compan...
 RRF Score: 0.01639 -> 6. COMMERCIAL PRICING PLANS
All billing is charged monthly or annually (with a 15% discoun...
 RRF Score: 0.01639 -> # Novasphere Dynamics - Product Changelog & Company News
Release Period: October 2026 (Ver...


> **What just happened?**
> RAG-Fusion reordered our candidate chunks using robust reciprocal rank consensus.

### Step 6: Pattern 3: Query Decomposition for Multi-Part Questions
When a question contains multiple distinct sub-questions, decomposing it allows retrieving specific chunks for each part.

In [7]:
decomp_prompt = ChatPromptTemplate.from_template(
    "Deconstruct this complex question into 2 simple sub-questions:\n"
    "Question: {question}\n"
    "Sub-questions (one per line):"
)

complex_q = "How much does the Enterprise plan cost, and what ROS2 versions are supported?"
sub_qs = [
    "How much does the Enterprise plan cost?",
    "What ROS2 versions are natively supported by Novasphere?"
]

sub_answers = []
for sq in sub_qs:
    relevant = vector_store.similarity_search(sq, k=1)[0]
    sub_answers.append(f"Sub-Q: {sq}\nEvidence: {relevant.page_content.strip()}")

print("\n---\n".join(sub_answers))

Sub-Q: How much does the Enterprise plan cost?
Evidence: - Custom Ultra-Tier Plan: $12,500 per month. Designed for fleets exceeding 200 nodes (up to 1,000 nodes). Includes custom on-premise edge deployments and dedicated solutions engineering.
---
Sub-Q: What ROS2 versions are natively supported by Novasphere?
Evidence: # Novasphere Dynamics - Product Changelog & Company News
Release Period: October 2026 (Version 4.8.0 Release & Announcements)
Published: October 28, 2026

## Version 4.8.0 Release Highlights (Engine Build 2026.10.480)


> **What just happened?**
> Each sub-question independently retrieved its corresponding source document.

### Step 7: Pattern 4: Step-Back Prompting for High-Level Concepts
Step-back prompting asks a broader, more abstract conceptual question to retrieve foundational context that grounds specific inquiries.

In [8]:
step_back_prompt = ChatPromptTemplate.from_template(
    "You are an AI assistant. Given this specific query, what is the broader, more abstract concept?\n"
    "Specific Query: {question}\n"
    "Step-back Query:"
)

specific_q = "Can I bring my golden retriever to the Zurich office on Friday?"
step_back_q = "What are the pet and animal policies for Novasphere office locations?"
print(f"Original:  {specific_q}")
print(f"Step-Back: {step_back_q}")

# Retrieve for both specific and step-back
combined_chunks = vector_store.similarity_search(specific_q, k=1) + vector_store.similarity_search(step_back_q, k=1)
print(f"\nCombined Context Retrieved ({len(combined_chunks)} chunks):")
for c in combined_chunks:
    print(f" - {c.page_content[:120].strip()}...")

Original:  Can I bring my golden retriever to the Zurich office on Friday?
Step-Back: What are the pet and animal policies for Novasphere office locations?

Combined Context Retrieved (2 chunks):
 - 3. EXPENSES, TRAVEL & LEARNING STIPEND
- Meal Expense Limit: Employees traveling on company business are reimbursed up t...
 - 3. Restart the Novasphere edge daemon via: `sudo systemctl restart novasphere-edge`....


> **What just happened?**
> Step-back retrieval gathered both specific facts and the overarching policy framework.

### Step 8: Pattern 5: Hypothetical Document Embeddings (HyDE)
HyDE asks an LLM to generate a hypothetical ideal document, then embeds that hypothetical document to search for real chunks with matching tone and structure.

In [9]:
hyde_prompt = ChatPromptTemplate.from_template(
    "Write a short, factual passage from a company handbook that answers this question:\n"
    "Question: {question}\n"
    "Passage:"
)

hypo_doc = "Novasphere allows domestic pets in the Zurich office on designated weekdays with facilities approval."
print("Generated Hypothetical Document (HyDE):")
print(f"'{hypo_doc}'")

# Search vector store using the hypothetical document embedding!
hyde_matches = vector_store.similarity_search(hypo_doc, k=2)
print("\nReal Chunks Retrieved via HyDE:")
for m in hyde_matches:
    print(f" - {m.page_content[:110].strip()}...")

Generated Hypothetical Document (HyDE):
'Novasphere allows domestic pets in the Zurich office on designated weekdays with facilities approval.'

Real Chunks Retrieved via HyDE:
 - ### What robotics middlewares are supported?
Native integration adapters are provided for:
1. ROS2 (Robot Oper...
 - # Novasphere Dynamics - Product Changelog & Company News
Release Period: October 2026 (Version 4.8.0 Release &...


> **What just happened?**
> HyDE matched document-to-document semantics rather than query-to-document semantics.

### Step 9: Architectural Trade-Off Comparison
Let's review the practical engineering trade-offs of each query translation strategy in production.

In [10]:
comparison_md = """
| Strategy        | Latency Overhead | Recall Boost | LLM Token Cost | Best Used When... |
|-----------------|------------------|--------------|----------------|-------------------|
| **Multi-Query** | Low (parallel)   | High         | 1 call         | Query phrasing is ambiguous or colloquial |
| **RAG-Fusion**  | Low (parallel)   | Very High    | 1 call + RRF   | Multiple valid search perspectives exist |
| **Decomposition**| Moderate       | High         | 1 call/sub-Q   | Complex, multi-hop or compound questions |
| **Step-Back**   | Low (parallel)   | High         | 1 call         | Query requires foundational principles or background |
| **HyDE**        | Moderate         | Moderate-High| 1 call         | User query is extremely brief or poorly structured |
"""
print(comparison_md)


| Strategy        | Latency Overhead | Recall Boost | LLM Token Cost | Best Used When... |
|-----------------|------------------|--------------|----------------|-------------------|
| **Multi-Query** | Low (parallel)   | High         | 1 call         | Query phrasing is ambiguous or colloquial |
| **RAG-Fusion**  | Low (parallel)   | Very High    | 1 call + RRF   | Multiple valid search perspectives exist |
| **Decomposition**| Moderate       | High         | 1 call/sub-Q   | Complex, multi-hop or compound questions |
| **Step-Back**   | Low (parallel)   | High         | 1 call         | Query requires foundational principles or background |
| **HyDE**        | Moderate         | Moderate-High| 1 call         | User query is extremely brief or poorly structured |



> **What just happened?**
> This comparison table guides your architecture choice based on latency and token budgets.

## 4. Try It Yourself!

### Exercises
1. **Exercise 1 (RRF Smoothing Constant):** Run `reciprocal_rank_fusion(results_by_query, k=10)` with $k=10$ instead of $k=60$. Does smaller $k$ give more or less weight to top-ranked documents?
2. **Exercise 2 (HyDE for Troubleshooting):** Create a hypothetical troubleshooting passage for *"ERR_QUANTUM_TIMEOUT"* and verify that HyDE retrieves the exact resolution steps.

In [11]:
# Solution 1: RRF with k=10
fused_k10 = reciprocal_rank_fusion(results_by_query, k=10)
print("Top chunk with k=10 (Score):", fused_k10[0][1])
print("Top chunk with k=60 (Score):", fused_results[0][1])
# Intuition: A smaller k (e.g. 10) increases the difference between rank 1 (1/11=0.09) and rank 2 (1/12=0.083), favoring top ranks more aggressively.

# Solution 2: HyDE for Troubleshooting
trouble_hypo = "To resolve ERR_QUANTUM_TIMEOUT, open port 9872 and check clock drift."
trouble_docs = vector_store.similarity_search(trouble_hypo, k=1)
print("\nTroubleshooting Doc retrieved via HyDE:")
print(trouble_docs[0].page_content[:120].strip())

Top chunk with k=10 (Score): 0.09090909090909091
Top chunk with k=60 (Score): 0.01639344262295082

Troubleshooting Doc retrieved via HyDE:
* Availability: 24 hours a day, 7 days a week, 365 days a year (24/7/365).
  * First Response Time Target: Within 15 min


## 5. Common Mistakes & Pitfalls
- **Using Multi-Query on Simple Lookups:** Multi-query triples retrieval load; only apply it to ambiguous or poorly formed questions.
- **Summing Raw Cosine Scores:** Never sum raw cosine similarity across different queries; use RRF ranks to eliminate score scale drift.
- **Hallucinated HyDE Embeddings:** If the LLM invents nonexistent entities in its hypothetical answer, HyDE may retrieve completely unrelated chunks.
- **Over-Decomposition:** Breaking simple questions into sub-questions adds unnecessary latency and token overhead.

---

## 6. Notebook Recap
- Query translation bridges the semantic gap between how humans ask questions and how documents are written.
- Multi-Query expands perspective; RAG-Fusion scores consensus using Reciprocal Rank Fusion (RRF).
- Decomposition breaks multi-hop queries into independent sub-queries.
- Step-Back prompting retrieves high-level principles to ground specific details.
- HyDE embeds a generated hypothetical answer to perform document-to-document search.


## 7. What's Next?

In **09_routing_and_query_construction.ipynb**, we learn how to route queries to specialized datasources dynamically and translate natural language into structured metadata filters.

## 8. Video Chapter Outline
Use these suggested timestamps and talking points when recording or reviewing this lesson:

- **00:00 - Why User Queries Fail:** The vocabulary mismatch problem in RAG.
- **02:15 - Multi-Query Perspective Expansion:** Generating diverse query angles with LLMs.
- **05:00 - The Math of Reciprocal Rank Fusion:** Hand-worked RRF example and scoring intuition.
- **08:30 - Query Decomposition:** Splitting compound questions into independent sub-queries.
- **11:15 - Step-Back Prompting:** Retrieving abstract principles alongside specific facts.
- **13:45 - Hypothetical Document Embeddings (HyDE):** Document-to-document vector search.
- **16:00 - Trade-Off Comparison Table:** Latency vs recall vs token cost across strategies.
